# 📊 Báo cáo Khoa học & Benchmark Pipeline (Milestone 1)

Notebook này phân tích dữ liệu trực tiếp từ hệ thống pipeline thu thập bài báo arXiv theo chuẩn Schema 1.2.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, 'src')
from arxiv_pipeline.statistics.service import StatisticsService
from arxiv_pipeline.benchmark.runner import BenchmarkService

stats_path = Path('statistics/baseline-001.json')
bench_path = Path('benchmarks/baseline-001.json')

with open(stats_path, 'r', encoding='utf-8') as f:
    stats = json.load(f)
with open(bench_path, 'r', encoding='utf-8') as f:
    bench = json.load(f)

summary = stats['summary']
print(f"Tổng số bài phát hiện: {summary['papers_discovered']}")
print(f"Số bài hoàn thành toàn diện: {summary['papers_complete']} ({summary['paper_success_rate'] or 0:.1%})")
print(f"Trích dẫn arXiv: {summary['references_arxiv_total']} (Tỷ lệ đủ metadata: {summary['reference_metadata_success_rate'] or 0:.1%})")

## 1. Trực quan hóa Tiến độ Thu thập theo Thành viên

In [ ]:
by_owner = stats.get('by_owner', {})
owners = sorted(by_owner.keys())
owner_labels = [o.capitalize() for o in owners]

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)
discovered = [by_owner[o].get('papers_discovered', 0) for o in owners]
complete = [by_owner[o].get('papers_complete', 0) for o in owners]

x = range(len(owners))
width = 0.35
ax.bar([i - width/2 for i in x], discovered, width, label='Phát hiện', color='#3498db')
ax.bar([i + width/2 for i in x], complete, width, label='Hoàn tất', color='#2ecc71')

ax.set_ylabel('Số lượng bài báo')
ax.set_title('Tiến độ Thu thập & Hoàn thiện theo Thành viên', fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(owner_labels)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Tối ưu hóa Dung lượng Nguồn TeX/BibTeX

In [ ]:
extracted = [by_owner[o].get('mean_extracted_bytes_before_filter') or 0 for o in owners]
retained = [by_owner[o].get('mean_retained_source_bytes') or 0 for o in owners]

extracted_kb = [e / 1024 for e in extracted]
retained_kb = [r / 1024 for r in retained]

fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
ax.bar([i - 0.15 for i in x], extracted_kb, 0.3, label='Trước lọc (Tổng giải nén)', color='#e74c3c')
ax.bar([i + 0.15 for i in x], retained_kb, 0.3, label='Sau lọc (Chỉ TeX/BibTeX)', color='#27ae60')

ax.set_ylabel('Dung lượng trung bình (KB)')
ax.set_title('Dung lượng Nguồn Trước vs. Sau khi Lọc File Phụ trợ', fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(owner_labels)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Hiệu năng & Tài nguyên Hệ thống

In [ ]:
peak_ram_mb = (bench.get('peak_ram_bytes') or 0) / (1024 * 1024)
mean_ram_mb = (bench.get('mean_ram_bytes') or 0) / (1024 * 1024)
wall_sec = bench.get('wall_seconds', 0.0)

print(f"Thời gian thực thi: {wall_sec:.2f}s")
print(f"Bộ nhớ RAM đỉnh: {peak_ram_mb:.2f} MB")
print(f"Bộ nhớ RAM trung bình: {mean_ram_mb:.2f} MB")
print(f"Phương pháp đo: {bench.get('method', 'tracemalloc')}")
print(f"Hệ điều hành: {bench.get('environment', {}).get('os', 'Unknown')}")
print(f"CPU: {bench.get('environment', {}).get('processor', 'Unknown')}")